In [32]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(sys.prefix).parent
%cd {PROJECT_ROOT}

/home/younes/younes/Projects/Python/barid_internship


In [210]:
import polars as pl

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
import calendar
from fpppy.utils import plot_series, plot_series_stacked, plot_diagnostics
from scipy.stats import pearsonr
from plotly import express as px
import altair as alt
from pathlib import Path
from lxml import html
from itertools import chain

import importlib
import offline_historique

importlib.reload(offline_historique)
from offline_historique import parse_historique_from_folder  # noqa: E402

cfg = pl.Config()
cfg.set_tbl_width_chars(10000)
cfg.set_fmt_str_lengths(100)
cfg.set_tbl_cols(-1)
cfg.set_tbl_rows(20)

polars.config.Config

In [170]:
def complete_operations(df):
    return df.filter(
        pl.col("status_code").first().over("id") == "depot",
        pl.col("status_code").last().over("id").is_in(["liv", "liv_ret"]),
    )


In [34]:
fields = pl.read_parquet("data/cab_dfs/fields.parquet")
operations = pl.read_parquet("data/cab_dfs/operations.parquet")
delivery = pl.read_parquet("data/cab_dfs/delivery.parquet")
services = pl.read_parquet("data/cab_dfs/services.parquet")

In [48]:
fields = fields
operations = operations.filter(is_valid="V")
delivery = delivery
services = services

In [66]:
print(operations)

shape: (856_005, 13)
┌──────────────────────┬──────────┬────────────────┬─────────────────────┬────────────────────────────────────────────────────────────┬────────────────────────────────┬────────────┬────────────────────────┬───────────┬───────────┬───────────┬─────────────┬──────────┐
│ cab                  ┆ id       ┆ Date_operation ┆ Heure_Syst_Oper     ┆ Statut                                                     ┆ Agence                         ┆ Agent_oper ┆ Etat                   ┆ Date_Etat ┆ Agent_maj ┆ ORIGINE   ┆ status_code ┆ is_valid │
│ ---                  ┆ ---      ┆ ---            ┆ ---                 ┆ ---                                                        ┆ ---                            ┆ ---        ┆ ---                    ┆ ---       ┆ ---       ┆ ---       ┆ ---         ┆ ---      │
│ str                  ┆ str      ┆ date           ┆ datetime[μs]        ┆ str                                                        ┆ str                            ┆ i64   

In [152]:
print(operations.select(pl.all().n_unique()))

shape: (1, 13)
┌────────┬────────┬────────────────┬─────────────────┬────────┬────────┬────────────┬──────┬───────────┬───────────┬─────────┬─────────────┬──────────┐
│ cab    ┆ id     ┆ Date_operation ┆ Heure_Syst_Oper ┆ Statut ┆ Agence ┆ Agent_oper ┆ Etat ┆ Date_Etat ┆ Agent_maj ┆ ORIGINE ┆ status_code ┆ is_valid │
│ ---    ┆ ---    ┆ ---            ┆ ---             ┆ ---    ┆ ---    ┆ ---        ┆ ---  ┆ ---       ┆ ---       ┆ ---     ┆ ---         ┆ ---      │
│ u32    ┆ u32    ┆ u32            ┆ u32             ┆ u32    ┆ u32    ┆ u32        ┆ u32  ┆ u32       ┆ u32       ┆ u32     ┆ u32         ┆ u32      │
╞════════╪════════╪════════════════╪═════════════════╪════════╪════════╪════════════╪══════╪═══════════╪═══════════╪═════════╪═════════════╪══════════╡
│ 104637 ┆ 111046 ┆ 859            ┆ 178563          ┆ 20     ┆ 1435   ┆ 4018       ┆ 14   ┆ 806       ┆ 2         ┆ 4       ┆ 14          ┆ 1        │
└────────┴────────┴────────────────┴─────────────────┴────────┴────────┴─

In [205]:
print(
    operations.group_by("status_code")
    .agg(pl.col("Statut").unique())
    .sort("status_code")
)

shape: (14, 2)
┌─────────────┬──────────────────────────────────────────────────────────────────────────────────────┐
│ status_code ┆ Statut                                                                               │
│ ---         ┆ ---                                                                                  │
│ str         ┆ list[str]                                                                            │
╞═════════════╪══════════════════════════════════════════════════════════════════════════════════════╡
│ aexp        ┆ ["Expédié vers"]                                                                     │
│ aff         ┆ ["Mis en distribution", "En attente de complément d'adresse/ou date de Rendez-vous"] │
│ affg        ┆ ["En instance au guichet", "À récupérer du guichet suite manque d'adresse"]          │
│ anoma       ┆ ["Mis en rebus"]                                                                     │
│ areturn     ┆ ["A Retourner"]                           

In [154]:
print(
    operations.group_by("id")
    .agg(pl.col("status_code").first())
    .group_by("status_code")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

shape: (7, 2)
┌─────────────┬────────┐
│ status_code ┆ count  │
│ ---         ┆ ---    │
│ str         ┆ u32    │
╞═════════════╪════════╡
│ depot       ┆ 104618 │
│ areturn     ┆ 6409   │
│ recpt       ┆ 13     │
│ lev         ┆ 3      │
│ aff         ┆ 1      │
│ aexp        ┆ 1      │
│ nrcl        ┆ 1      │
└─────────────┴────────┘


In [141]:
(
    operations.group_by("id")
    .having(pl.col("status_code").first() == "areturn")
    .agg(
        start=pl.col("Heure_Syst_Oper").first(),
        end=pl.col("Heure_Syst_Oper").last(),
    )
    .with_columns((pl.col("end") - pl.col("start")).dt.total_days().alias("lead_days"))
    .with_columns((pl.col("lead_days") // 1 * 1).alias("bucket"))
    .group_by("bucket")
    .agg(pl.len().alias("count"))
    .sort("bucket")
    .with_columns(cum_count=pl.col("count").cum_sum(reverse=True))
)

bucket,count,cum_count
i64,u32,u32
0,491,6409
1,277,5918
2,624,5641
3,671,5017
4,711,4346
…,…,…
251,1,5
304,1,4
369,1,3


In [187]:
pdf = (
    operations.pipe(complete_operations)
    .group_by("id")
    .agg(
        start=pl.col("Heure_Syst_Oper").first(),
        end=pl.col("Heure_Syst_Oper").last(),
    )
    .with_columns((pl.col("end") - pl.col("start")).dt.total_days().alias("lead_days"))
    .with_columns((pl.col("lead_days") // 1 * 1).alias("bucket_days"))
    .group_by("bucket_days")
    .agg(pl.len().alias("count"))
    .sort("bucket_days")
    .with_columns(
        cum_count=pl.col("count").cum_sum(reverse=True),
        ratio=pl.col("count") / pl.col("count").sum(),
        cum_ratio=pl.col("count").cum_sum(reverse=True) / pl.col("count").sum(),
    )
    .filter(pl.col("cum_ratio").ge(0.001))
    .to_pandas()
)


chart = (
    alt.Chart(pdf)
    .mark_line(point=True)
    .encode(
        x=alt.X("bucket_days:Q", title="Lead Time Bucket (days)").scale(
            domainMin=1,
            type="log",
        ),
        y=alt.Y("cum_ratio:Q", title="cum_ratio").scale(
            # domainMin=0.001,
            type="log",
        ),
        tooltip=[
            alt.Tooltip("bucket_days:Q", title="days"),
            alt.Tooltip("cum_ratio:Q", title="cum_ratio"),
            alt.Tooltip("count:Q", title="count"),
        ],
    )
    .properties(
        width=900, height=400, title="Aggregated Delivery Lead Time Distribution"
    )
)

chart

alt.Chart(...)

In [191]:
edges: pl.DataFrame = (
    operations.pipe(complete_operations)
    .with_columns(
        [
            pl.col("status_code").shift(-1).over("cab").alias("next"),
            pl.col("Heure_Syst_Oper").shift(-1).over("cab").alias("next_ts"),
            pl.col("Agence").shift(-1).over("cab").alias("next_Agence"),
        ]
    )
    .filter(pl.col("next").is_not_null())
    .with_columns(
        (pl.col("next_ts") - pl.col("Heure_Syst_Oper")).dt.total_hours().alias("hours")
    )
)

In [212]:
print(edges.group_by("status_code", "next").len().sort("len", descending=True).head(20))

shape: (20, 3)
┌─────────────┬─────────┬────────┐
│ status_code ┆ next    ┆ len    │
│ ---         ┆ ---     ┆ ---    │
│ str         ┆ str     ┆ u32    │
╞═════════════╪═════════╪════════╡
│ recpt       ┆ aexp    ┆ 168083 │
│ aexp        ┆ recpt   ┆ 149945 │
│ aff         ┆ liv     ┆ 86651  │
│ depot       ┆ recpt   ┆ 84777  │
│ recpt       ┆ aff     ┆ 80093  │
│ aexp        ┆ aff     ┆ 17548  │
│ depot       ┆ lev     ┆ 16806  │
│ lev         ┆ recpt   ┆ 15757  │
│ affg        ┆ liv     ┆ 12711  │
│ aff         ┆ chrgctr ┆ 12121  │
│ recpt       ┆ affg    ┆ 11510  │
│ chrgctr     ┆ aff     ┆ 8843   │
│ aexp        ┆ affg    ┆ 4714   │
│ chrgctr     ┆ chrgctr ┆ 3682   │
│ aexp        ┆ aexp    ┆ 3063   │
│ nrcl        ┆ recpt   ┆ 3050   │
│ aff         ┆ liv_ret ┆ 2761   │
│ recpt       ┆ recpt   ┆ 2317   │
│ affg        ┆ nrcl    ┆ 2315   │
│ chrgctr     ┆ recpt   ┆ 2182   │
└─────────────┴─────────┴────────┘


In [169]:
print(edges.filter(status_code="recpt").group_by("next").all())


shape: (10, 17)
┌─────────┬──────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────┬────────────────────────────────────────┬───────────────────────────────────────────────────────────────────┬────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────┬────────────────────────────────────────────────────────────────────────────┬────────────────┬───────────────────┬───────────────────────────────────────┬───────────────────────────────┬───────────────────┬───────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────┐
│ next    ┆ cab                                                                      ┆ id                                     ┆ Date_operation                         ┆ Heure_